# Khmer-LLaDA Small-A (74.0M) — training on Kaggle (dual T4)

Clones [`Pich09/Khmer-LLaDA-Small`](https://github.com/Pich09/Khmer-LLaDA-Small) and runs the Small-A (74.0M) pretraining
pipeline end to end: tokenizer + corpus -> packed shards -> unit tests ->
Warmup-Stable-Decay training via the repo's native `torchrun` DDP support -> loss curves ->
sample generations -> (optional) push to the Hub.

**Before you run — Notebook settings (right sidebar):**
1. **Accelerator -> GPU T4 x2** — both GPUs are used (`torchrun --nproc_per_node=2`).
2. **Internet -> On** (clone + tokenizer/corpus downloads).
3. *(optional, for cross-session resume)* **Add-ons -> Secrets -> add `HF_TOKEN`** (write-scoped
   for `Panhapich/Khmer-LLaDA-Small-A`) — checkpoints then sync automatically; without it they stay local to
   the session (or fall back to a `/kaggle/input` dataset from a prior session).

If a session times out mid-run, just **Run All** again — the resume cell picks up
`checkpoints/last.pt` (locally, or pulled from `HUB_CKPT_REPO`) and training continues from
there; the LR schedule and step count are correct either way regardless of how many GPUs a
given session has (see `training/train.py::lr_at`).

## 1 · Environment

In [ ]:
import os, sys, subprocess, platform, torch

print('python :', platform.python_version())
print('torch  :', torch.__version__)
print('cuda   :', torch.cuda.is_available())
N_GPU = torch.cuda.device_count()
print('n_gpu  :', N_GPU)
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'  gpu{i}: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{p.major}{p.minor}')
if N_GPU == 0:
    print('WARNING: no GPU — turn on the accelerator, or expect training to be unusably slow.')
elif N_GPU == 1:
    print('NOTE: only 1 GPU visible — training cell below falls back to nproc_per_node=1 automatically.')

WORK     = '/kaggle/working'
REPO_DIR = os.path.join(WORK, 'Khmer-LLaDA-Small')

## 2 · Clone the repo

In [ ]:
REPO_URL = 'https://github.com/Pich09/Khmer-LLaDA-Small.git'

if not os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=False)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print('cwd:', os.getcwd())
print(subprocess.run(['git', '-C', REPO_DIR, 'log', '--oneline', '-1'],
                     capture_output=True, text=True).stdout.strip())

## 3 · Dependencies

`torch` / `numpy` / `pandas` / `tqdm` / `matplotlib` / `pyyaml` ship with the Kaggle image.
Only the tokenizer stack (+ `datasets` for the extra raw corpora, Small-A only) is added.

In [ ]:
# critical for training (SentencePiece over the pre-segmented corpus)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece==0.2.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'huggingface_hub>=0.24,<1.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'datasets==2.21.0', 'pyarrow'], check=True)

# best-effort: only used by the raw-prompt path in the sampling cell
try:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'khmer-nltk==1.6', 'wordninja==2.0.0'], check=True)
    RAW_TOKENIZER_OK = True
except subprocess.CalledProcessError as e:
    print('khmer-nltk/wordninja install failed (sampling with raw-text prompts will be skipped):', e)
    RAW_TOKENIZER_OK = False

## 4 · Hugging Face token + checkpoint repo (optional)

The tokenizer and corpus are public, so training works with **no token**. A **write-scoped**
`HF_TOKEN` (Kaggle -> Add-ons -> Secrets) plus a `HUB_CKPT_REPO` you can write to enables
cross-session persistence: every checkpoint sync (cell near the bottom) pushes to
`checkpoints/` in that HF model repo, and the resume cell below pulls the latest back.
Without a token, checkpoints live only in `/kaggle/working` — use *Save Version* to keep them.

In [ ]:
HF_TOKEN = ''
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or ''
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get('HF_TOKEN', '')

HUB_CKPT_REPO = 'Panhapich/Khmer-LLaDA-Small-A'   # HF model repo checkpoints sync to when HF_TOKEN is set (write access needed)

HUB_ACTIVE = False
if HF_TOKEN:
    from huggingface_hub import login, create_repo
    login(token=HF_TOKEN, add_to_git_credential=False)
    if HUB_CKPT_REPO:
        try:
            create_repo(HUB_CKPT_REPO, repo_type='model', exist_ok=True, token=HF_TOKEN)
            HUB_ACTIVE = True
            print('HF: logged in, checkpoint hub ->', HUB_CKPT_REPO)
        except Exception as e:
            print('HF: could not create/access', HUB_CKPT_REPO, '-', e)
    else:
        print('HF: logged in, but HUB_CKPT_REPO is empty — checkpoints stay local this session.')
else:
    print('no HF_TOKEN — tokenizer/corpus downloads use anonymous access; checkpoints stay local.')

## 5 · Tokenizer

Downloads the shared `khmer-sp-8k` SentencePiece model into `<repo>/tokenizer/` and asserts
it is exactly **vocab 8000 with specials `<PAD>=0 <UNK>=1 <BOS>=2 <EOS>=3 <MASK>=4`** — the
values every config and `khmer_llada/constants.py` hardcode; a mismatch here would corrupt
training silently.

In [ ]:
from huggingface_hub import hf_hub_download

TOK_DIR = os.path.join(REPO_DIR, 'tokenizer')
os.makedirs(TOK_DIR, exist_ok=True)

for f in ['khmer_sp.model', 'khmer_sp.vocab', 'khmer_segmentation.py',
          'gazetteer.json', 'latin_exceptions.json', 'tokenizer_info.json', 'USAGE.md']:
    try:
        hf_hub_download('Panhapich/khmer-sp-8k', f, repo_type='model',
                        local_dir=TOK_DIR, token=HF_TOKEN or None)
    except Exception as e:
        print(f'  (skip {f}: {e})')
assert os.path.exists(os.path.join(TOK_DIR, 'khmer_sp.model')), 'khmer_sp.model missing — cannot continue'
print('tokenizer:', sorted(os.listdir(TOK_DIR)))

import sentencepiece as spm
_sp = spm.SentencePieceProcessor(model_file=os.path.join(TOK_DIR, 'khmer_sp.model'))
TOK_VOCAB   = _sp.get_piece_size()
TOK_SPECIAL = [_sp.piece_to_id(t) for t in ('<PAD>', '<UNK>', '<BOS>', '<EOS>', '<MASK>')]
print(f'tokenizer vocab: {TOK_VOCAB}   PAD/UNK/BOS/EOS/MASK ids: {TOK_SPECIAL}')
assert TOK_VOCAB == 8000, f'khmer-sp-8k should be vocab 8000, got {TOK_VOCAB}'
assert TOK_SPECIAL == [0, 1, 2, 3, 4], 'special-token ids differ from the fixed 0..4 the repo hardcodes'

## 6 · Original corpus

Pulls the pre-segmented base corpus (already khmer-nltk word-segmented, so a bare
SentencePiece pass in step 7 is correct **and** fast).

In [ ]:
RAW_DIR = os.path.join(REPO_DIR, 'data', 'raw')
os.makedirs(RAW_DIR, exist_ok=True)

CORPUS_FILE = os.path.join(RAW_DIR, 'all_text_segmented.txt')
if not os.path.exists(CORPUS_FILE):
    hf_hub_download('Panhapich/khmer-text-corpus', 'all_text_segmented.txt',
                    repo_type='dataset', local_dir=RAW_DIR, token=HF_TOKEN or None)

n_lines = sum(1 for _ in open(CORPUS_FILE, encoding='utf-8'))
print(f'corpus: {CORPUS_FILE}  {os.path.getsize(CORPUS_FILE)/1e6:.0f} MB  {n_lines:,} lines')

## 7 · Extra raw corpora (nphearum + Khmer Wikipedia)

Small-A (74.0M) trains on the **combined** corpus (base + these two sources, ~407.8M tokens total)
— same corpus and token budget as the other size, so the two runs differ only in
architecture. These sources are NOT pre-segmented, so they go through
`extract_extra_corpus.py` then `pretokenize_extra_parallel.py` (khmer-nltk word
segmentation, parallelized across cores) in the next step — appended as extra TRAIN shards
after the base corpus's. The base corpus's val split (step 6) is left untouched, so
`val_nll_bound` stays comparable across the whole run history.

In [ ]:
import time
from datasets import load_dataset

def _load_dataset_retry(*args, attempts=5, **kwargs):
    # These are multi-hundred-MB to multi-GB downloads over a plain (non-resumable at this
    # layer) HTTP stream; a single dropped connection anywhere in that transfer raises
    # ChunkedEncodingError/IncompleteRead with no retry of its own, killing the whole cell.
    # Retry with backoff instead of letting one hiccup abort the run.
    for attempt in range(1, attempts + 1):
        try:
            return load_dataset(*args, **kwargs)
        except Exception as e:
            if attempt == attempts:
                raise
            wait = min(60, 5 * 2 ** (attempt - 1))
            print(f'  download attempt {attempt}/{attempts} failed ({type(e).__name__}: {e}) '
                  f'— retrying in {wait}s', flush=True)
            time.sleep(wait)

RAW_EXTRA = os.path.join(REPO_DIR, 'data', 'raw_extra')
NPHEARUM_DIR = os.path.join(RAW_EXTRA, 'nphearum')
WIKI_DIR = os.path.join(RAW_EXTRA, 'wikipedia_km', '20231101.km')
os.makedirs(NPHEARUM_DIR, exist_ok=True)
os.makedirs(WIKI_DIR, exist_ok=True)

nphearum_pq = os.path.join(NPHEARUM_DIR, 'train.parquet')
if not os.path.exists(nphearum_pq):
    ds = _load_dataset_retry('nphearum/khmer-raw-text-3M-v2', split='train')
    ds.to_parquet(nphearum_pq)
    print('nphearum:', len(ds), 'rows')

wiki_pq = os.path.join(WIKI_DIR, 'train-00000-of-00001.parquet')
if not os.path.exists(wiki_pq):
    ds = _load_dataset_retry('wikimedia/wikipedia', '20231101.km', split='train')
    ds.to_parquet(wiki_pq)
    print('wikipedia_km:', len(ds), 'rows')

combined_raw = os.path.join(RAW_EXTRA, 'combined_raw.txt')
if not os.path.exists(combined_raw):
    subprocess.run([sys.executable, 'scripts/extract_extra_corpus.py'], check=True)
print(combined_raw, f'{os.path.getsize(combined_raw)/1e6:.0f} MB')

## 8 · Pre-tokenize -> packed `uint16` shards

In [ ]:
SEQ_LEN   = 512
SHARD_DIR = os.path.join(REPO_DIR, 'data', 'shards')

have_base = os.path.isdir(SHARD_DIR) and any(
    n.startswith('train_') and n.endswith('.npy') for n in os.listdir(SHARD_DIR))
if not have_base:
    subprocess.run([sys.executable, 'scripts/pretokenize.py',
                    '--in', CORPUS_FILE, '--seq-len', str(SEQ_LEN),
                    '--out-dir', SHARD_DIR], check=True)
else:
    print('base shards already present — skipping pretokenize.py')

import glob
n_extra_shards = len([p for p in glob.glob(os.path.join(SHARD_DIR, 'train_*.npy'))])
have_extra_marker = os.path.exists(os.path.join(RAW_EXTRA, '.pretokenized'))
if not have_extra_marker:
    subprocess.run([sys.executable, 'scripts/pretokenize_extra_parallel.py',
                    '--in', combined_raw, '--seq-len', str(SEQ_LEN),
                    '--out-dir', SHARD_DIR, '--workers', str(os.cpu_count() or 4)], check=True)
    open(os.path.join(RAW_EXTRA, '.pretokenized'), 'w').close()
else:
    print('extra shards already present — skipping pretokenize_extra_parallel.py')

import numpy as np
def _tok_count(split):
    return sum(np.load(p, mmap_mode='r').shape[0]
               for p in glob.glob(os.path.join(SHARD_DIR, f'{split}_*.npy'))) * SEQ_LEN
train_tok, val_tok = _tok_count('train'), _tok_count('val')
print(f'packed train tokens: {train_tok:,}   val tokens: {val_tok:,}')
assert train_tok > 0 and val_tok > 0, 'pre-tokenization produced no data'

## 9 · Unit tests (Milestone-1)

GPU-free tests assert bidirectional attention, the `1/t` diffusion loss, bit-exact
checkpoint resume, and the tokenizer round-trip. (The separate overfit gate —
`training/overfit.py` on `configs/tiny.json` — is a useful correctness check when
iterating on the loss/mask/attention code locally, but needs ~4000 steps to hit its
target and is a hard `check=True` stop; it's intentionally left out of this notebook's
critical path so a slow or borderline overfit run can't block the real training cell
below. Run it manually first if you've changed anything in `khmer_llada/`.)

In [ ]:
subprocess.run([sys.executable, '-m', 'pytest', '-q',
                'tests/test_bidirectional.py', 'tests/test_diffusion.py',
                'tests/test_resume.py', 'tests/test_tokenizer.py'], check=True)

## 10 · Resume check

Finds a checkpoint to continue from, in order: (1) `checkpoints/last.pt` already in this
session (re-running cells without restarting), (2) `checkpoints/last.pt` in `HUB_CKPT_REPO`
— the normal cross-session path when `HF_TOKEN` is set, (3) the newest `last.pt` under
`/kaggle/input/**` — a **previous session's notebook output** added as an input dataset
(Kaggle -> Add Input -> Notebook Output -> this notebook's latest version), the no-token
fallback for resuming across sessions.

In [ ]:
MODEL_CFG = 'configs/small_a.json'
TRAIN_CFG = 'configs/train_t4.yaml'
RUN_NAME  = 'small_a_run1'
CKPT_DIR  = os.path.join(REPO_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)

import glob, shutil

RESUME = None
local_last = os.path.join(CKPT_DIR, 'last.pt')
if os.path.exists(local_last):
    RESUME = local_last
    print('resuming from this session:', local_last)
elif HUB_ACTIVE:
    try:
        from huggingface_hub import hf_hub_download
        p = hf_hub_download(HUB_CKPT_REPO, 'checkpoints/last.pt', repo_type='model', token=HF_TOKEN)
        shutil.copy(p, local_last)
        for nm in ('best.pt', 'status.json'):
            try:
                p2 = hf_hub_download(HUB_CKPT_REPO, f'checkpoints/{nm}', repo_type='model', token=HF_TOKEN)
                shutil.copy(p2, os.path.join(CKPT_DIR, nm))
            except Exception:
                pass
        RESUME = local_last
        print('resuming from hub:', HUB_CKPT_REPO)
    except Exception as e:
        print('no checkpoint on hub yet — starting from scratch:', e)
if RESUME is None:
    # no-token fallback: a prior session's Notebook Output added as an Input dataset
    candidates = sorted(glob.glob('/kaggle/input/**/last.pt', recursive=True),
                        key=os.path.getmtime, reverse=True)
    if candidates:
        src_dir = os.path.dirname(candidates[0])
        for nm in ('last.pt', 'best.pt', 'status.json'):
            p = os.path.join(src_dir, nm)
            if os.path.exists(p):
                shutil.copy(p, os.path.join(CKPT_DIR, nm))
        RESUME = local_last
        print('resuming from Kaggle input dataset:', candidates[0])
    else:
        print('no local / hub / input checkpoint — starting from scratch. '
              'For hands-free cross-session resume, add an HF_TOKEN secret above (write '
              'access to HUB_CKPT_REPO); otherwise use Save Version each session and '
              'Add Input -> Notebook Output next time.')

## 11 · Train

Launches the repo's own `training/train.py` via `torchrun` — Warmup-Stable-Decay schedule,
fp16 + `GradScaler`, grad-clip + grad-accum, `best.pt` on every `val_nll_bound` improvement,
`last.pt` + `status.json` every `save_every_steps`. The schedule is keyed on tokens actually
consumed (not raw step count), so it stays correct even if a later session runs with a
different number of GPUs than this one. If the session times out, just re-run the notebook —
the resume cell above picks `last.pt` back up.

In [ ]:
assert RESUME is None or isinstance(RESUME, str), (
    f'RESUME must be a checkpoint path or None, got {RESUME!r} ({type(RESUME).__name__}) — '
    'check the Resume check cell above for a stray reassignment.')

cmd = ['torchrun', '--standalone', f'--nproc_per_node={max(1, N_GPU)}',
       'training/train.py', '--model-config', MODEL_CFG, '--train-config', TRAIN_CFG]
if RESUME:
    cmd += ['--resume', RESUME]
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)

## 12 · Loss curves

In [ ]:
import json, matplotlib.pyplot as plt

mfile = os.path.join(REPO_DIR, 'experiments', RUN_NAME, 'metrics.jsonl')
rows  = [json.loads(l) for l in open(mfile)] if os.path.exists(mfile) else []
tr = [(r['step'], r['train_loss'])    for r in rows if 'train_loss' in r]
vl = [(r['step'], r['val_nll_bound']) for r in rows if 'val_nll_bound' in r]
lr = [(r['step'], r['lr'])            for r in rows if 'lr' in r]

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
if tr: ax[0].plot(*zip(*tr)); ax[0].set(title='train_loss', xlabel='step')
if vl: ax[1].plot(*zip(*vl), color='tab:red'); ax[1].set(title='val_nll_bound', xlabel='step')
if lr: ax[2].plot(*zip(*lr), color='tab:green'); ax[2].set(title='lr (WSD schedule)', xlabel='step')
plt.tight_layout(); plt.show()

status_path = os.path.join(CKPT_DIR, 'status.json')
if os.path.exists(status_path):
    print(json.dumps(json.load(open(status_path)), indent=2))

## 13 · Sample from the checkpoint

Uses the repo's own `evaluation/generate_eval.py` (picks `best.pt`, else `last.pt`).

In [ ]:
ckpt_path = next((os.path.join(CKPT_DIR, n) for n in ('best.pt', 'last.pt')
                  if os.path.exists(os.path.join(CKPT_DIR, n))), '')
assert ckpt_path, 'no checkpoint in ' + CKPT_DIR

prompts_path = os.path.join(WORK, 'sample_prompts.txt')
with open(prompts_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(['', 'ប្រទេសកម្ពុជា', 'នៅថ្ងៃនេះ', 'បញ្ញាសិប្បនិម្មិត', 'ការសិក្សា']))

subprocess.run([sys.executable, 'evaluation/generate_eval.py',
                '--ckpt', ckpt_path, '--model-config', MODEL_CFG,
                '--prompts-file', prompts_path, '--steps', '128',
                '--out', os.path.join(WORK, 'generations.json')], check=True)
print(json.dumps(json.load(open(os.path.join(WORK, 'generations.json'))), ensure_ascii=False, indent=2))

## 14 · Sync checkpoints to the Hub (cross-session persistence)

Skipped automatically if `HUB_CKPT_REPO` (cell 4) was left empty.

In [ ]:
if HUB_ACTIVE:
    from huggingface_hub import HfApi
    HfApi().upload_folder(folder_path=CKPT_DIR, path_in_repo='checkpoints',
                          repo_id=HUB_CKPT_REPO, repo_type='model', token=HF_TOKEN,
                          commit_message=f'sync @ {RUN_NAME}')
    print('synced', CKPT_DIR, '->', HUB_CKPT_REPO + '/checkpoints/')
else:
    print('no HF_TOKEN / HUB_CKPT_REPO — use Save Version to persist /kaggle/working instead.')

## 15 · Export for downstream use (optional)

Packs a portable model-only directory (`model.pt` + `config.json` + `meta.json` + tokenizer)
and, if `--repo` is reachable with `HF_TOKEN`, pushes it to the Hub.

In [ ]:
cmd = [sys.executable, 'scripts/export_and_push.py',
       '--ckpt', ckpt_path, '--export-dir', 'export/small_a_run1_pretrained']
if not (HF_TOKEN and HUB_CKPT_REPO):
    cmd.append('--no-push')
else:
    cmd += ['--repo', HUB_CKPT_REPO]
subprocess.run(cmd, check=True)

## Notes

- **Overfit gate is a hard stop.** If that cell raises, the model/loss is wrong — fix before
  spending GPU hours.
- **`val_nll_bound`** is a Monte-Carlo upper bound (fixed seed), not a true perplexity —
  comparable across checkpoints and against a future AR baseline, not against standard LM
  perplexity.
- **LR schedule is tokens-based** (`training/train.py::lr_at`), so switching GPU count
  between sessions (e.g. this notebook running with 1 GPU one day, 2 the next) does not
  corrupt the Warmup-Stable-Decay schedule the way a step-count-based schedule would.
- A Kaggle session is ~12h; a full run may span several sessions — set `HUB_CKPT_REPO` above
  for hands-free cross-session resume, or use *Save Version* each session otherwise.